# BBC News Headlines: Preprocessing, POS Tagging & NER (spaCy)

This notebook walks through a realistic NLP workflow on **BBC news headlines** (`bbc_news.csv`, 1,000 rows):

1. **Clean and tokenize** headlines (pandas + NLTK)
2. **Part-of-speech (POS) tagging** — label each token as noun, verb, adjective, etc.
3. **Named entity recognition (NER)** — detect people, places, organizations, dates

We use **spaCy** for tagging and NER (industry-standard, fast, accurate). For a lighter NLTK-only NER demo, see `06-named-entity-recognition.ipynb`.

**Prerequisites:** `01-nlp-fundamentals.ipynb` for preprocessing concepts; `09-tripadvisor-preprocessing-practical.ipynb` for a similar step-by-step clean-only walkthrough.

## Setup

Install spaCy and the small English model once (from the repo root):

```bash
python -m pip install spacy
python -m spacy download en_core_web_sm
```

In [ ]:
import re

import nltk
import pandas as pd
import spacy
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

from nlp_helpers import DATASETS_DIR, download_nltk_data

download_nltk_data()

try:
    nlp = spacy.load("en_core_web_sm")
except OSError as e:
    raise OSError(
        "spaCy model not found. Run: python -m spacy download en_core_web_sm"
    ) from e

lemmatizer = WordNetLemmatizer()
STOPWORDS = set(stopwords.words("english"))
print("Ready.")

## 1. Load and explore the data

Each row is a BBC article with metadata. We focus on **`title`** — short, rich text for POS/NER practice.

In [ ]:
bbc = pd.read_csv(f"{DATASETS_DIR}/bbc_news.csv")
bbc = bbc.drop(columns=["Unnamed: 0"], errors="ignore")

print("Shape:", bbc.shape)
print("Columns:", bbc.columns.tolist())
bbc.info()
bbc[["title", "description"]].head(3)

## 2. Preprocess headlines

Pipeline applied to each title:

| Step | Why |
|------|-----|
| Lowercase | Normalize casing |
| Remove stopwords | Drop noise (`the`, `a`, …) |
| Remove punctuation | Keep word tokens only |
| Tokenize | Split into words (NLTK `word_tokenize`) |
| Lemmatize | `studies` → `study` (dictionary base form) |

We use a **function** instead of many dataframe columns — easier to read and reuse. The tripadvisor notebook (`09-…`) shows the same steps column-by-column for teaching.

In [ ]:
def preprocess_title(title: str) -> list[str]:
    """Return lemmatized tokens for one headline."""
    text = str(title).lower()
    text = " ".join(w for w in text.split() if w not in STOPWORDS)
    text = re.sub(r"[^\w\s]", "", text)
    tokens = word_tokenize(text)
    return [lemmatizer.lemmatize(t) for t in tokens if t]


titles = bbc[["title"]].copy()
titles["tokens"] = titles["title"].apply(preprocess_title)

print("Example title:", titles.loc[0, "title"])
print("Tokens:", titles.loc[0, "tokens"][:15], "...")

In [ ]:
# Flatten all headline tokens into one list for corpus-level analysis
all_tokens = sum(titles["tokens"], [])
print(f"{len(titles)} headlines → {len(all_tokens)} tokens")

## 3. Part-of-speech (POS) tagging

**POS tagging** assigns a grammatical label to each token (e.g. `NOUN`, `VERB`, `ADJ`).

- **NLTK** can tag with `nltk.pos_tag` (used in `nlp_helpers.preprocess_text`).
- **spaCy** is usually preferred in production: one pipeline, fast, consistent with NER.

We run spaCy on the joined token string (one `Doc` object). In real projects you would call `nlp(headline)` per document or use `nlp.pipe()` for batches.

In [ ]:
corpus_text = " ".join(all_tokens)
doc = nlp(corpus_text)

# Build a tidy table (avoid slow pd.concat in a loop)
pos_df = pd.DataFrame(
    [
        {"token": token.text, "pos_tag": token.pos_}
        for token in doc
        if not token.is_space and not token.is_punct
    ]
)

print(f"Tagged {len(pos_df)} tokens")
pos_df.head(10)

In [ ]:
# Most common tokens by POS — value_counts is clearer than manual groupby slices
def top_by_pos(tag: str, n: int = 10) -> pd.Series:
    subset = pos_df.loc[pos_df["pos_tag"] == tag, "token"]
    return subset.value_counts().head(n)


print("Top nouns in headlines:")
display_nouns = top_by_pos("NOUN")
print(display_nouns)

print("\nTop verbs:")
print(top_by_pos("VERB"))

print("\nTop adjectives:")
print(top_by_pos("ADJ"))

**Reading the results:** Frequent nouns (`uk`, `year`, `world`) reflect news themes. Verbs often describe events (`say`, `get`). Adjectives add tone (`new`, `first`).

spaCy also provides **fine-grained** tags (`token.tag_`, Penn Treebank) and **dependencies** (`token.dep_`) if you need deeper linguistic features later.

## 4. Named entity recognition (NER)

**NER** finds spans of text that refer to real-world objects:

| Label | Meaning | Example |
|-------|---------|--------|
| `PERSON` | People | Liz Truss |
| `GPE` | Countries, cities | London, UK |
| `ORG` | Companies, teams | BBC, Red Bull |
| `DATE` | Dates | October 2022 |

Use `doc.ents` — each item is a **span** (multi-word entity), not a single token. The original pattern of looping `doc.ents` as if they were tokens is a common mistake; we extract `ent.text` and `ent.label_` directly.

In [ ]:
# NER on original headlines preserves capitalization (helps detect "UK", "Biden")
headline_docs = list(nlp.pipe(bbc["title"].head(200), batch_size=50))

ner_rows = [
    {"text": ent.text, "label": ent.label_}
    for spacy_doc in headline_docs
    for ent in spacy_doc.ents
]
ner_df = pd.DataFrame(ner_rows)

print(f"Entities in first 200 headlines: {len(ner_df)}")
ner_df.head(10)

In [ ]:
def top_entities(label: str, n: int = 10) -> pd.DataFrame:
    subset = ner_df.loc[ner_df["label"] == label]
    counts = subset["text"].value_counts().head(n).reset_index()
    counts.columns = ["entity", "count"]
    return counts


print("Most mentioned people:")
print(top_entities("PERSON"))

print("\nMost mentioned places (GPE):")
print(top_entities("GPE"))

print("\nMost mentioned organizations:")
print(top_entities("ORG"))

## 5. Tag one headline end-to-end

Inspect a single title in the interactive spaCy style (`token.text`, POS, entity label).

In [ ]:
sample_title = bbc.loc[1, "title"]
sample_doc = nlp(sample_title)

print("Title:", sample_title)
print("\nTokens (text → POS):")
for token in sample_doc:
    if not token.is_space:
        print(f"  {token.text:20} {token.pos_}")

print("\nEntities:")
for ent in sample_doc.ents:
    print(f"  {ent.label_:10} {ent.text}")

## Summary

| Task | Tool used | Takeaway |
|------|-----------|----------|
| Cleaning | pandas + NLTK | Functions > many temp columns for production code |
| POS | spaCy `token.pos_` | Find dominant nouns/verbs in a corpus |
| NER | spaCy `doc.ents` | Extract people, places, orgs from headlines |
| Scale | `nlp.pipe()` | Batch processing for many documents |

**Next steps:** Topic trends (`05-topic-modeling-lda.ipynb`), classification (`02-…`), or larger models (BERT) for state-of-the-art NER.